In [3]:
#BUILT-IN
!pip install langchain langchain-core langchain-community pydantic ddgs langchain_experimental


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('weather today in india')
print(results)

India Weather Today: India braces for wet week ahead; rains disrupt life in Tripura, Bengaluru ... The weather agency, as per a report in The Indian ... India Weather Today: Light to moderate rainfall, thunderstorms, and strong winds likely in parts of India this week According to experts quoted by news agency PTI , temperatures may even scale to 47 degrees Celsius in parts of northwest India. Weather forecast for today India ... In New Delhi, at the moment, fog is rolling in, covering streets and highways in a thick haze. Today, the highest temperature in Mumbai will be 30°C , while the lowest temperature will be 23°C . ... In Mumbai, today, there will be 11h and ...


In [7]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [9]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()
results = shell_tool.invoke('whoami')
print(results)

Executing command:
 whoami
azuread\pranitabhatt



In [ ]:
#CUSTOM TOOLS
#using @ tool
from langchain_core.tools import tool

In [11]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [12]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [13]:
# Step 3 - add tool decorator

@tool #will make it a special function and help to communicate with LLM
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [14]:
result=multiply.invoke({'a':10,'b':2})

In [15]:
print(result)

20


In [16]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [ ]:
print(multiply.args_schema.model_json_schema()) #this is what llm will see

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [22]:
#Using Structured Tool and Pydantic
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [23]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

C:\Users\PranitaBhatt\AppData\Local\Temp\ipykernel_8728\3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\PranitaBhatt\AppData\Local\Temp\ipykernel_8728\3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [24]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [ ]:

multiply_tool = StructuredTool.from_function(
    func=multiply_func, #multiply function
    name="multiply", 
    description="Multiply two numbers",
    args_schema=MultiplyInput #pydantic class
)

In [26]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


In [27]:
#BUILT-IN
#Using base tool
from langchain.tools import BaseTool
from typing import Type

In [28]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

C:\Users\PranitaBhatt\AppData\Local\Temp\ipykernel_8728\908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\PranitaBhatt\AppData\Local\Temp\ipykernel_8728\908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [29]:
class MultiplyTool(BaseTool): #multiplytool class is inheritng basetool class
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput #a pydantic shcema

    def _run(self, a: int, b: int) -> int:
        return a * b

In [30]:
multiply_tool = MultiplyTool() #create an object

In [31]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}
